# Torso Parallel mechanism 

## Forward Kinematics

In [21]:
import sympy as sp
import numpy as np
import kinematics_utils as ku

ls, lc, lr, lo, ld = sp.symbols('ls lc lr lo ld', positive=True, real=True)
a, b = sp.symbols('a b', real=True)
theta1, theta2 = sp.symbols('theta1 theta2', real=True)
w_a, w_b = sp.symbols('w_alpha w_alpha', real=True)
w_theta1, w_theta2 = sp.symbols('w_theta1 w_theta2', real=True)

def Rx(theta):
    c, s = sp.cos(theta), sp.sin(theta)
    return sp.Matrix([[1, 0, 0], [0, c, -s], [0, s, c]])


def Ry(theta):
    c, s = sp.cos(theta), sp.sin(theta)
    return sp.Matrix([[c, 0, s], [0, 1, 0], [-s, 0, c]])

def clean(expr):
    # expr = sp.expand_trig(expr)
    # expr = sp.together(expr)
    expr = sp.cancel(expr)
    expr = sp.trigsimp(expr)
    expr = sp.simplify(expr)
    return expr

# Home positions
r_o_a2_home = sp.Matrix([ 0, ls/2,  ld])
r_o_b2_home = sp.Matrix([ lc,ls/2,  ld])
r_o_c2_home = sp.Matrix([ lc,ls/2, -lo])

r_o_a3_home = sp.Matrix([ 0, -ls/2,  ld])
r_o_b3_home = sp.Matrix([ lc,-ls/2,  ld])
r_o_c3_home = sp.Matrix([ lc,-ls/2, -lo])


### Base → Torso rotation

In [22]:

rot_base_torso = Ry(b) * Rx(a)

# Intermediate position
r_o_a2_int = rot_base_torso * r_o_a2_home 
r_o_b2_int = rot_base_torso * r_o_b2_home
r_o_a3_int = rot_base_torso * r_o_a3_home 
r_o_b3_int = rot_base_torso * r_o_b3_home


### Loop closure equation - Derivation

In [23]:

# ===================  Constraint equation ||R1 - Rot(theta1)*R2||^2 = lr^2 =================

motor_axis = r_o_a2_int - r_o_a3_int
u = motor_axis / motor_axis.norm()

# Vectors R1, R2
R1 = r_o_c2_home - r_o_a2_int
R2 = r_o_b2_int - r_o_a2_int

P1 = r_o_c3_home - r_o_a3_int
P2 = r_o_b3_int - r_o_a3_int

R2_prll = u * (u.dot(R2))
R2_perp = R2 - R2_prll
P2_prll = u * (u.dot(P2))
P2_perp = P2 - P2_prll

### Loop Closure Equations

In [24]:
# eq1 = ((R1 - R2_prll - R2_perp*sp.cos(theta1) - u.cross(R2)*sp.sin(theta1)).norm())**2 - lr**2
# eq2 = ((P1 - P2_prll - P2_perp*sp.cos(theta2) - u.cross(P2)*sp.sin(theta2)).norm())**2 - lr**2

i = (R1 - R2_prll - R2_perp*sp.cos(theta1) - u.cross(R2)*sp.sin(theta1))
j = (P1 - P2_prll - P2_perp*sp.cos(theta2) - u.cross(P2)*sp.sin(theta2))

eq1 = i.dot(i) - lr**2
eq2 = j.dot(j) - lr**2

# eq1 =(-2*lc**2*sp.cos(a + theta1) + 2*lc**2 - 2*lc*ld*sp.sin(a) -
#     2*lc*ld*sp.sin(theta1) - 2*lc*lo*sp.sin(a + theta1)*sp.cos(b) +
#     lc*ls*sp.sin(b)*sp.sin(a + theta1) + ld**2 + 2*ld*lo*sp.cos(a)*sp.cos(b) -
#     ld*ls*sp.sin(b)*sp.cos(a) + lo**2 - lo*ls*sp.sin(b) - lr**2 - ls**2*sp.cos(b)/2 + ls**2/2)

# eq2 =(-2*lc**2*sp.cos(a + theta2) + 2*lc**2 - 2*lc*ld*sp.sin(a) -
#     2*lc*ld*sp.sin(theta2) - 2*lc*lo*sp.sin(a + theta2)*sp.cos(b) -
#     lc*ls*sp.sin(b)*sp.sin(a + theta2) + ld**2 + 2*ld*lo*sp.cos(a)*sp.cos(b) +
#     ld*ls*sp.sin(b)*sp.cos(a) + lo**2 + lo*ls*sp.sin(b) - lr**2 - ls**2*sp.cos(b)/2 + ls**2/2)


In [25]:
clean(eq1)

2*lc**2*sin(b)*sin(theta1)*cos(a) - 2*lc**2*cos(b)*cos(theta1) + 2*lc**2 - 2*lc*ld*sin(b)*cos(a) - 2*lc*ld*sin(theta1) - 2*lc*lo*sin(b)*cos(theta1) - 2*lc*lo*sin(theta1)*cos(a)*cos(b) - lc*ls*sin(a)*sin(b) - lc*ls*sin(a)*sin(theta1) + ld**2 + 2*ld*lo*cos(a)*cos(b) + ld*ls*sin(a) + lo**2 + lo*ls*sin(a)*cos(b) - lr**2 - ls**2*cos(a)/2 + ls**2/2

In [26]:
clean(eq2)

2*lc**2*sin(b)*sin(theta2)*cos(a) - 2*lc**2*cos(b)*cos(theta2) + 2*lc**2 - 2*lc*ld*sin(b)*cos(a) - 2*lc*ld*sin(theta2) - 2*lc*lo*sin(b)*cos(theta2) - 2*lc*lo*sin(theta2)*cos(a)*cos(b) + lc*ls*sin(a)*sin(b) + lc*ls*sin(a)*sin(theta2) + ld**2 + 2*ld*lo*cos(a)*cos(b) - ld*ls*sin(a) + lo**2 - lo*ls*sin(a)*cos(b) - lr**2 - ls**2*cos(a)/2 + ls**2/2

In [27]:
# clean(eq1 + eq2)

In [28]:
# clean(eq1 - eq2)


## Analytical Jacobian

In [29]:
f = sp.Matrix([clean(eq1), clean(eq2)])
variables = sp.Matrix([a,b])

In [30]:
J = clean(f).jacobian(variables)
clean(J)

Matrix([
[-2*lc**2*sin(a)*sin(b)*sin(theta1) + 2*lc*ld*sin(a)*sin(b) + 2*lc*lo*sin(a)*sin(theta1)*cos(b) - lc*ls*sin(b)*cos(a) - lc*ls*sin(theta1)*cos(a) - 2*ld*lo*sin(a)*cos(b) + ld*ls*cos(a) + lo*ls*cos(a)*cos(b) + ls**2*sin(a)/2, 2*lc**2*sin(b)*cos(theta1) + 2*lc**2*sin(theta1)*cos(a)*cos(b) - 2*lc*ld*cos(a)*cos(b) + 2*lc*lo*sin(b)*sin(theta1)*cos(a) - 2*lc*lo*cos(b)*cos(theta1) - lc*ls*sin(a)*cos(b) - 2*ld*lo*sin(b)*cos(a) - lo*ls*sin(a)*sin(b)],
[-2*lc**2*sin(a)*sin(b)*sin(theta2) + 2*lc*ld*sin(a)*sin(b) + 2*lc*lo*sin(a)*sin(theta2)*cos(b) + lc*ls*sin(b)*cos(a) + lc*ls*sin(theta2)*cos(a) - 2*ld*lo*sin(a)*cos(b) - ld*ls*cos(a) - lo*ls*cos(a)*cos(b) + ls**2*sin(a)/2, 2*lc**2*sin(b)*cos(theta2) + 2*lc**2*sin(theta2)*cos(a)*cos(b) - 2*lc*ld*cos(a)*cos(b) + 2*lc*lo*sin(b)*sin(theta2)*cos(a) - 2*lc*lo*cos(b)*cos(theta2) + lc*ls*sin(a)*cos(b) - 2*ld*lo*sin(b)*cos(a) + lo*ls*sin(a)*sin(b)]])

In [31]:
detJ = clean(J.det())
print(clean(detJ))

4*lc**4*sin(a)*sin(theta1 - theta2)*cos(b)**2 - 4*lc**4*sin(a)*sin(theta1 - theta2) + 4*lc**3*ld*sin(a)*cos(b)**2*cos(theta1) - 4*lc**3*ld*sin(a)*cos(b)**2*cos(theta2) - 4*lc**3*ld*sin(a)*cos(theta1) + 4*lc**3*ld*sin(a)*cos(theta2) + 8*lc**3*lo*sin(a)*sin(b)*sin(theta1 - theta2)*cos(b) - 2*lc**3*ls*sin(b)*sin(theta1 + theta2)*cos(a) - 4*lc**3*ls*sin(theta1)*sin(theta2)*cos(a)**2*cos(b) + 2*lc**3*ls*cos(a)*cos(b)**2*cos(theta1) + 2*lc**3*ls*cos(a)*cos(b)**2*cos(theta2) - 2*lc**3*ls*cos(a)*cos(theta1) - 2*lc**3*ls*cos(a)*cos(theta2) - lc**3*ls*cos(2*b - theta1)/2 + lc**3*ls*cos(2*b + theta1)/2 - lc**3*ls*cos(2*b - theta2)/2 + lc**3*ls*cos(2*b + theta2)/2 + 8*lc**2*ld*lo*sin(a)*sin(b)*cos(b)*cos(theta1) - 8*lc**2*ld*lo*sin(a)*sin(b)*cos(b)*cos(theta2) + 2*lc**2*ld*ls*sin(b)*cos(a)*cos(theta1) + 2*lc**2*ld*ls*sin(b)*cos(a)*cos(theta2) + 2*lc**2*ld*ls*sin(2*b) + 4*lc**2*ld*ls*sin(theta1)*cos(a)**2*cos(b) + 4*lc**2*ld*ls*sin(theta2)*cos(a)**2*cos(b) - 4*lc**2*lo**2*sin(a)*sin(theta1 - theta2

In [32]:
values = {
    ls: 0.08,
    lc: 0.075,
    lr: 0.115,
    lo: 0.025,
    ld: 0.09,
    # theta1: np.radians(0.0),
    # theta2: np.radians(0.0)
    # theta1: np.radians(-10.22956697),
    # theta2: np.radians(-10.22956697)
    # theta1: np.radians(-5.55843853),
    # theta2: np.radians(5.10457742)
    theta1: np.radians(-10.0),
    theta2: np.radians(-10.0)
}

### Solving Numerically using Newton Raphson (nsolve)

In [33]:
eq1_num, eq2_num = [i.subs(values) for i in (eq1, eq2)]

alpha_beta = sp.nsolve(
    [eq1_num, eq2_num],
    [a, b],
    [np.radians(9.99), np.radians(9.99)],
    tol=1e-15,
    # verbose=True
)

alpha_beta_rad = np.array(alpha_beta, dtype=float)
alpha_beta_deg = np.degrees(alpha_beta_rad.astype(float))

In [34]:
alpha_beta_rad

array([[3.24858997e-23],
       [1.70701623e-01]])

In [35]:
alpha_beta_deg

array([[1.86130494e-21],
       [9.78048254e+00]])